# MODULE 8 — Domain-Generalized ConvoReleNet for Subject-Independent MI-EEG

**Purpose:** train the proposed high-accuracy architecture on the frozen Module 5/6/7 interface without changing the validated preprocessing pipeline.

### Frozen project specification
- Input: `(N, 22, 640)` EEG epochs
- Sampling rate: 160 Hz
- Primary preprocessing: 8–30 Hz, continuous preprocessing, 4 s epochs
- Classes: `left`, `right`, `feet`
- 118 subjects / 9,316 epochs in the validated cache
- Source-only robust normalization
- Strict subject-wise LOSO and cross-dataset zero-calibration protocols

### Proposed model
**Multi-scale temporal CNN → spatial CNN → Transformer relational encoder → attention pooling → 128-D embedding → classifier**

Training losses: **cross-entropy + center loss + subject-adversarial domain loss**.

The recurrent branch is intentionally disabled by default because the uploaded 2026 ConvoReleNet study found the additional LSTM variant did not improve IV-2a accuracy; transfer learning and conservative fine-tuning were the strongest consistently supported components. The same paper reported 79.44 ± 11.09% for compact transfer learning and 87.55 ± 9.64% for its Tanh variant on IV-2a.

> This notebook keeps strict zero-calibration evaluation separate from optional few-shot adaptation. Target test epochs are never used for normalization, training, hyperparameter selection, or threshold selection in the strict modes.

In [1]:
# ============================================================
# CELL 1 — IMPORTS, REPRODUCIBILITY, DEVICE, PATH DISCOVERY
# ============================================================
from __future__ import annotations

import os
import json
import math
import time
import copy
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")

SEED = 20260824
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# ------------------------------------------------------------
# Robust project-root discovery for the user's MacBook setup.
# ------------------------------------------------------------
candidates = []
if os.environ.get("CROSS_DATASET_MI_PROJECT_ROOT"):
    candidates.append(Path(os.environ["CROSS_DATASET_MI_PROJECT_ROOT"]).expanduser())
candidates.extend([
    Path.home() / "Project2" / "cross_dataset_mi_project",
    Path.cwd() / "cross_dataset_mi_project",
    Path.home() / "cross_dataset_mi_project",
])

PROJECT_ROOT = next((p for p in candidates if p.exists()), candidates[0])
MANIFEST_ROOT = PROJECT_ROOT / "manifests"
CACHE_ROOT = PROJECT_ROOT / "cache"
RESULTS_ROOT = PROJECT_ROOT / "results"
MODULE8_ROOT = RESULTS_ROOT / "module_8_dg_convorelenet"
CHECKPOINT_ROOT = MODULE8_ROOT / "checkpoints"
HISTORY_ROOT = MODULE8_ROOT / "histories"
PRED_ROOT = MODULE8_ROOT / "predictions"

for p in [MODULE8_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, PRED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CACHE_PATH = CACHE_ROOT / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
CACHE_META_PATH = MANIFEST_ROOT / "module_6_cache_metadata.csv"
WITHIN_LOSO_PATH = MANIFEST_ROOT / "module_6_within_dataset_loso_folds.csv"
TRANSFER_PATH = MANIFEST_ROOT / "module_6_cross_dataset_transfer_folds.csv"
PROTOCOL_PATH = MANIFEST_ROOT / "module_6_baseline_protocol.json"

REQUIRED = [CACHE_PATH, CACHE_META_PATH, WITHIN_LOSO_PATH, TRANSFER_PATH, PROTOCOL_PATH]

print("=" * 78)
print("MODULE 8 — DG-CONVORELENET")
print("=" * 78)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device      :", DEVICE)
print("PyTorch     :", torch.__version__)
print("Seed        :", SEED)
for p in REQUIRED:
    print(("✓ " if p.exists() else "✗ ") + str(p))
assert all(p.exists() for p in REQUIRED), "One or more Module 6 artifacts are missing."

MODULE 8 — DG-CONVORELENET
PROJECT_ROOT: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project
Device      : mps
PyTorch     : 2.10.0
Seed        : 20260824
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cache_metadata.csv
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_within_dataset_loso_folds.csv
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cross_dataset_transfer_folds.csv
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_baseline_protocol.json


In [2]:
# ============================================================
# CELL 2 — FROZEN DATA SPECIFICATION + EXPERIMENT CONTROLS
# ============================================================
PRIMARY_CLASSES = ["left", "right", "feet"]
CLASS_TO_ID = {"left": 0, "right": 1, "feet": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

N_CHANNELS = 22
N_SAMPLES = 640
N_CLASSES = 3
TARGET_SFREQ = 160.0

# ------------------------------------------------------------
# Model: high-accuracy, compact CNN + Transformer core.
# Multi-scale temporal kernels mirror the recommendation to
# learn short/mid/long temporal structure before relational
# attention. Tanh is the default activation because it was the
# strongest activation for four-class IV-2a in the uploaded
# 2026 transfer-learning paper.
# ------------------------------------------------------------
@dataclass
class Config:
    # Reproducibility
    seed: int = SEED

    # Training
    batch_size: int = 64
    max_epochs: int = 50
    warmup_epochs: int = 5
    lr: float = 3e-4
    min_lr: float = 1e-5
    weight_decay: float = 1e-4
    patience: int = 10
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    # Loss weights
    center_loss_weight: float = 0.01
    domain_loss_weight: float = 0.05
    center_loss_lr: float = 0.5
    grl_max_lambda: float = 0.20

    # Model
    temporal_channels_each: int = 16
    fusion_channels: int = 48
    spatial_channels: int = 64
    token_dim: int = 128
    transformer_layers: int = 4
    transformer_heads: int = 8
    transformer_ffn: int = 256
    transformer_dropout: float = 0.15
    embedding_dim: int = 128
    classifier_dropout: float = 0.25
    use_lstm: bool = False

    # EEG-safe train augmentation only; OFF for validation/test.
    train_noise_std: float = 0.008
    train_amp_jitter: float = 0.05
    train_channel_dropout: float = 0.05
    train_time_mask_prob: float = 0.15
    train_time_mask_max: int = 32

    # Evaluation
    validation_subject_fraction: float = 0.15
    validation_seed_offset: int = 900

    # Operational controls
    run_within_loso: bool = True
    run_cross_dataset: bool = True
    smoke_test: bool = False
    smoke_folds_per_protocol: int = 1
    resume: bool = True
    save_predictions: bool = True
    plot_histories: bool = True

CFG = Config()

print(json.dumps(asdict(CFG), indent=2))

{
  "seed": 20260824,
  "batch_size": 64,
  "max_epochs": 50,
  "warmup_epochs": 5,
  "lr": 0.0003,
  "min_lr": 1e-05,
  "weight_decay": 0.0001,
  "patience": 10,
  "grad_clip": 1.0,
  "label_smoothing": 0.05,
  "center_loss_weight": 0.01,
  "domain_loss_weight": 0.05,
  "center_loss_lr": 0.5,
  "grl_max_lambda": 0.2,
  "temporal_channels_each": 16,
  "fusion_channels": 48,
  "spatial_channels": 64,
  "token_dim": 128,
  "transformer_layers": 4,
  "transformer_heads": 8,
  "transformer_ffn": 256,
  "transformer_dropout": 0.15,
  "embedding_dim": 128,
  "classifier_dropout": 0.25,
  "use_lstm": false,
  "train_noise_std": 0.008,
  "train_amp_jitter": 0.05,
  "train_channel_dropout": 0.05,
  "train_time_mask_prob": 0.15,
  "train_time_mask_max": 32,
  "validation_subject_fraction": 0.15,
  "validation_seed_offset": 900,
  "run_within_loso": true,
  "run_cross_dataset": true,
  "smoke_test": false,
  "smoke_folds_per_protocol": 1,
  "resume": true,
  "save_predictions": true,
  "plot_hist

In [3]:
# ============================================================
# CELL 3 — LOAD FROZEN METADATA + MANIFESTS
# ============================================================
cache_meta_df = pd.read_csv(CACHE_META_PATH)
within_loso_df = pd.read_csv(WITHIN_LOSO_PATH)
transfer_df = pd.read_csv(TRANSFER_PATH)

assert len(cache_meta_df) == 9316, len(cache_meta_df)
assert cache_meta_df["subject"].nunique() == 118
assert set(cache_meta_df["harmonized_class"].unique()) == set(PRIMARY_CLASSES)
assert len(within_loso_df) == 118
assert len(transfer_df) == 118

print("Cache epochs       :", len(cache_meta_df))
print("Unique subjects    :", cache_meta_df["subject"].nunique())
print("Within LOSO folds  :", len(within_loso_df))
print("Transfer folds     :", len(transfer_df))
print("Classes            :", PRIMARY_CLASSES)
print("Class counts:")
print(cache_meta_df["harmonized_class"].value_counts().sort_index())

Cache epochs       : 9316
Unique subjects    : 118
Within LOSO folds  : 118
Transfer folds     : 118
Classes            : ['left', 'right', 'feet']
Class counts:
harmonized_class
feet     3103
left     3127
right    3086
Name: count, dtype: int64


In [4]:
# ============================================================
# CELL 4 — HDF5 INDEXED STORE + SOURCE-ONLY ROBUST NORMALIZER
# ============================================================
class HDF5Store:
    def __init__(self, path: Path):
        self.path = Path(path)
        self.h5 = None

    def __enter__(self):
        self.h5 = h5py.File(self.path, "r")
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.h5 is not None:
            self.h5.close()
        self.h5 = None

    def get_X(self, indices):
        indices = np.asarray(indices, dtype=np.int64)
        if len(indices) == 0:
            return np.empty((0, N_CHANNELS, N_SAMPLES), dtype=np.float32)
        # h5py fancy indexing expects monotonic indices; restore original order.
        order = np.argsort(indices)
        sorted_idx = indices[order]
        X_sorted = self.h5["X"][sorted_idx]
        inverse = np.argsort(order)
        return np.asarray(X_sorted[inverse], dtype=np.float32)


class SourceOnlyRobustNormalizer:
    """Exact channel-wise median/IQR fit on source training epochs only."""
    def __init__(self, eps=1e-6):
        self.eps = float(eps)
        self.median_ = None
        self.iqr_ = None
        self.fitted_subjects_ = tuple()

    def fit(self, X_source, source_subjects):
        X_source = np.asarray(X_source, dtype=np.float64)
        source_subjects = [str(s) for s in source_subjects]
        assert X_source.ndim == 3 and X_source.shape[1:] == (22, 640)
        assert len(source_subjects) == len(X_source)
        values = X_source.transpose(1, 0, 2).reshape(22, -1)
        self.median_ = np.median(values, axis=1)
        q25 = np.percentile(values, 25, axis=1)
        q75 = np.percentile(values, 75, axis=1)
        self.iqr_ = np.maximum(q75 - q25, self.eps)
        self.fitted_subjects_ = tuple(sorted(set(source_subjects)))
        return self

    def transform(self, X):
        if self.median_ is None:
            raise RuntimeError("Normalizer has not been fitted.")
        X = np.asarray(X, dtype=np.float32)
        return ((X - self.median_[None, :, None]) / self.iqr_[None, :, None]).astype(np.float32)

    def assert_target_excluded(self, target_subject):
        if str(target_subject) in self.fitted_subjects_:
            raise AssertionError(f"Normalization leakage: target {target_subject} was used in fit.")


def compute_class_ids(meta):
    return np.asarray([CLASS_TO_ID[x] for x in meta["harmonized_class"]], dtype=np.int64)


def safe_indices(json_string):
    return np.asarray(json.loads(json_string), dtype=np.int64)

print("Indexed HDF5 store + source-only normalizer: ready")

Indexed HDF5 store + source-only normalizer: ready


In [5]:
# ============================================================
# CELL 5 — GROUPED SOURCE TRAIN/VALIDATION SPLIT
# ============================================================

def grouped_source_split(source_indices, meta, target_subject=None, seed=0, val_fraction=0.15):
    source_indices = np.asarray(source_indices, dtype=np.int64)
    source_meta = meta.loc[source_indices].copy()

    if target_subject is not None:
        assert str(target_subject) not in set(source_meta["subject"].astype(str))

    subjects = np.array(sorted(source_meta["subject"].astype(str).unique()))
    rng = np.random.default_rng(seed)
    shuffled = subjects.copy()
    rng.shuffle(shuffled)
    n_val = max(1, int(round(len(shuffled) * val_fraction)))
    n_val = min(n_val, max(1, len(shuffled) - 1))
    val_subjects = set(shuffled[:n_val])

    val_mask = source_meta["subject"].astype(str).isin(val_subjects).to_numpy()
    train_idx = source_indices[~val_mask]
    val_idx = source_indices[val_mask]

    assert len(train_idx) > 0 and len(val_idx) > 0
    assert set(meta.loc[train_idx, "subject"]).isdisjoint(set(meta.loc[val_idx, "subject"]))
    if target_subject is not None:
        assert str(target_subject) not in set(meta.loc[train_idx, "subject"].astype(str))
        assert str(target_subject) not in set(meta.loc[val_idx, "subject"].astype(str))

    return train_idx, val_idx, sorted(val_subjects)

In [6]:
# ============================================================
# CELL 6 — EEG-SAFE TRAIN AUGMENTATION
# ============================================================
class EEGAugment:
    def __init__(self, noise_std=0.008, amp_jitter=0.05, channel_dropout=0.05,
                 time_mask_prob=0.15, time_mask_max=32):
        self.noise_std = float(noise_std)
        self.amp_jitter = float(amp_jitter)
        self.channel_dropout = float(channel_dropout)
        self.time_mask_prob = float(time_mask_prob)
        self.time_mask_max = int(time_mask_max)

    def __call__(self, x):
        # x: torch [C,T], source-train only.
        x = x.clone()
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        if self.amp_jitter > 0:
            scale = 1.0 + (torch.rand((x.shape[0], 1), device=x.device) * 2 - 1) * self.amp_jitter
            x = x * scale
        if self.channel_dropout > 0 and torch.rand(()) < self.channel_dropout:
            c = int(torch.randint(0, x.shape[0], (1,)).item())
            x[c] = 0.0
        if self.time_mask_prob > 0 and torch.rand(()) < self.time_mask_prob:
            width = int(torch.randint(8, max(9, self.time_mask_max + 1), (1,)).item())
            start = int(torch.randint(0, max(1, x.shape[1] - width + 1), (1,)).item())
            x[:, start:start+width] = 0.0
        return x


class ArrayEEGDataset(Dataset):
    def __init__(self, X, y, domains, augment=None):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.domains = np.asarray(domains, dtype=np.int64)
        self.augment = augment
        assert self.X.shape[1:] == (22, 640)
        assert len(self.X) == len(self.y) == len(self.domains)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        if self.augment is not None:
            x = self.augment(x)
        return x, torch.tensor(self.y[idx], dtype=torch.long), torch.tensor(self.domains[idx], dtype=torch.long)

In [7]:
# ============================================================
# CELL 7 — GRADIENT REVERSAL, CENTER LOSS, ATTENTION POOLING
# ============================================================
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd):
    return GradientReversalFn.apply(x, lambd)


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim, lr=0.5):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim) * 0.02)
        self.lr = float(lr)

    def forward(self, features, labels):
        centers_batch = self.centers.index_select(0, labels)
        return 0.5 * ((features - centers_batch) ** 2).sum(dim=1).mean()

    @torch.no_grad()
    def update_centers(self, features, labels):
        # Optional explicit center update in addition to the optimizer. Disabled
        # by default; the optimizer receives gradients through the normal loss.
        return


class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.Tanh(),
            nn.Linear(dim // 2, 1),
        )

    def forward(self, tokens):
        # tokens: [B,L,D]
        weights = torch.softmax(self.score(tokens).squeeze(-1), dim=1)
        pooled = torch.sum(tokens * weights.unsqueeze(-1), dim=1)
        return pooled, weights

In [8]:
# ============================================================
# CELL 8 — PROPOSED DG-CONVORELENET ARCHITECTURE
# ============================================================
class TemporalBranch(nn.Module):
    def __init__(self, out_channels, kernel_size, activation=nn.Tanh()):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, out_channels, kernel_size=(1, kernel_size), padding=(0, kernel_size // 2), bias=False),
            nn.BatchNorm2d(out_channels),
            activation,
            nn.Dropout2d(0.10),
        )

    def forward(self, x):
        return self.net(x)


class DGConvoReleNet(nn.Module):
    """
    Proposed model:
        multi-scale temporal CNN
        -> spatial convolution across all 22 channels
        -> temporal patch embedding
        -> Transformer relational encoder
        -> attentive pooling
        -> 128-D discriminative embedding
        -> class head + subject-domain head

    The optional LSTM is kept available for ablation, but defaults to False
    because the uploaded 2026 study reported no gain from its recurrent extension.
    """
    def __init__(self, num_classes=3, num_domains=2, cfg=CFG):
        super().__init__()
        act = nn.Tanh
        branch_kernels = [15, 31, 63]
        self.temporal = nn.ModuleList([
            TemporalBranch(cfg.temporal_channels_each, k, activation=act())
            for k in branch_kernels
        ])

        self.spatial = nn.Sequential(
            nn.Conv2d(cfg.fusion_channels, cfg.spatial_channels, kernel_size=(22, 1), bias=False),
            nn.BatchNorm2d(cfg.spatial_channels),
            act(),
            nn.Dropout2d(0.10),
        )

        self.patch = nn.Sequential(
            nn.Conv2d(cfg.spatial_channels, cfg.token_dim, kernel_size=(1, 8), stride=(1, 8), bias=False),
            nn.BatchNorm2d(cfg.token_dim),
            act(),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.token_dim,
            nhead=cfg.transformer_heads,
            dim_feedforward=cfg.transformer_ffn,
            dropout=cfg.transformer_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.transformer_layers)

        self.positional = nn.Parameter(torch.zeros(1, 80, cfg.token_dim))
        nn.init.normal_(self.positional, std=0.02)

        self.attn_pool = AttentionPooling(cfg.token_dim)

        self.embedding = nn.Sequential(
            nn.Linear(cfg.token_dim, cfg.embedding_dim),
            act(),
            nn.LayerNorm(cfg.embedding_dim),
            nn.Dropout(cfg.classifier_dropout),
        )

        self.classifier = nn.Linear(cfg.embedding_dim, num_classes)

        self.domain_classifier = nn.Sequential(
            nn.Linear(cfg.embedding_dim, 64),
            act(),
            nn.Dropout(0.20),
            nn.Linear(64, num_domains),
        )

        self.use_lstm = cfg.use_lstm
        if self.use_lstm:
            self.lstm = nn.LSTM(cfg.token_dim, cfg.token_dim // 2, num_layers=1, batch_first=True, bidirectional=True)
        else:
            self.lstm = None

    def forward(self, x, grl_lambda=0.0, return_tokens=False):
        # Input [B,C,T] -> [B,1,C,T]
        x = x.unsqueeze(1)
        branches = [b(x) for b in self.temporal]
        x = torch.cat(branches, dim=1)
        x = self.spatial(x)
        x = self.patch(x).squeeze(2).transpose(1, 2)  # [B,L,D]

        L = x.shape[1]
        pos = self.positional[:, :L]
        x = x + pos
        x = self.transformer(x)

        if self.lstm is not None:
            x, _ = self.lstm(x)

        pooled, weights = self.attn_pool(x)
        emb = self.embedding(pooled)
        logits = self.classifier(emb)
        dom_logits = self.domain_classifier(grad_reverse(emb, grl_lambda))

        out = {
            "logits": logits,
            "domain_logits": dom_logits,
            "embedding": emb,
            "attention": weights,
        }
        if return_tokens:
            out["tokens"] = x
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# smoke model just to validate dimensions
_model = DGConvoReleNet(num_classes=3, num_domains=10, cfg=CFG).to(DEVICE)
with torch.no_grad():
    _out = _model(torch.randn(2, 22, 640, device=DEVICE))
print("Trainable parameters:", f"{count_parameters(_model):,}")
print("logits shape       :", tuple(_out["logits"].shape))
print("embedding shape    :", tuple(_out["embedding"].shape))
print("domain logits shape:", tuple(_out["domain_logits"].shape))
del _model, _out

Trainable parameters: 709,886
logits shape       : (2, 3)
embedding shape    : (2, 128)
domain logits shape: (2, 10)


In [9]:
# ============================================================
# CELL 9 — TRAINING UTILITIES
# ============================================================
def make_domain_ids(meta_subset):
    subjects = meta_subset["subject"].astype(str).tolist()
    unique = {s: i for i, s in enumerate(sorted(set(subjects)))}
    domain_ids = np.asarray([unique[s] for s in subjects], dtype=np.int64)
    return domain_ids, unique


def make_loaders(X_train, y_train, d_train, X_val, y_val, d_val, cfg):
    aug = EEGAugment(
        cfg.train_noise_std,
        cfg.train_amp_jitter,
        cfg.train_channel_dropout,
        cfg.train_time_mask_prob,
        cfg.train_time_mask_max,
    )
    train_ds = ArrayEEGDataset(X_train, y_train, d_train, augment=aug)
    val_ds = ArrayEEGDataset(X_val, y_val, d_val, augment=None)

    # Subject-balanced sampling prevents subjects with more trials from
    # dominating the feature extractor and domain-adversarial objective.
    counts = np.bincount(d_train, minlength=max(1, int(d_train.max()) + 1)).astype(np.float64)
    counts[counts == 0] = 1.0
    sample_weights = 1.0 / counts[d_train]
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )

    pin = DEVICE.type == "cuda"
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, sampler=sampler, shuffle=False,
        num_workers=0, pin_memory=pin, drop_last=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
        num_workers=0, pin_memory=pin
    )
    return train_loader, val_loader


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def scheduled_grl_lambda(epoch, max_epochs, max_lambda):
    # Smooth DANN-style ramp: 0 -> max_lambda.
    p = epoch / max(1, max_epochs - 1)
    return max_lambda * (2.0 / (1.0 + math.exp(-10.0 * (p - 0.5))) - 1.0)


def compute_all_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion_matrix": json.dumps(cm.tolist()),
    }

In [10]:
# ============================================================
# CELL 10 — ONE-FOLD TRAINER
# ============================================================
def train_one_fold(X_source, y_source, domain_source, X_val, y_val, domain_val,
                   num_domains, cfg, seed, fold_tag):
    set_seed(seed)

    train_loader, val_loader = make_loaders(
        X_source, y_source, domain_source,
        X_val, y_val, domain_val,
        cfg,
    )

    model = DGConvoReleNet(
        num_classes=N_CLASSES,
        num_domains=num_domains,
        cfg=cfg,
    ).to(DEVICE)

    center = CenterLoss(N_CLASSES, cfg.embedding_dim, lr=cfg.center_loss_lr).to(DEVICE)

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(center.parameters()),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg.max_epochs,
        eta_min=cfg.min_lr,
    )

    ce = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    dom_ce = nn.CrossEntropyLoss()

    best_state = None
    best_center = None
    best_val_score = -np.inf
    best_val_loss = np.inf
    wait = 0
    history = []

    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        center.train()
        grl_lambda = 0.0 if epoch <= cfg.warmup_epochs else scheduled_grl_lambda(epoch - cfg.warmup_epochs, max(1, cfg.max_epochs - cfg.warmup_epochs), cfg.grl_max_lambda)

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb, db in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            db = db.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            out = model(xb, grl_lambda=grl_lambda)
            cls_loss = ce(out["logits"], yb)
            c_loss = center(out["embedding"], yb)
            d_loss = dom_ce(out["domain_logits"], db) if num_domains > 1 else torch.zeros((), device=DEVICE)
            loss = cls_loss + cfg.center_loss_weight * c_loss + cfg.domain_loss_weight * d_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

            train_loss_sum += float(loss.detach().cpu()) * len(xb)
            train_correct += int((out["logits"].argmax(1) == yb).sum().detach().cpu())
            train_total += len(xb)

        scheduler.step()

        # Validation
        model.eval()
        center.eval()
        val_loss_sum = 0.0
        y_true = []
        y_pred = []
        with torch.no_grad():
            for xb, yb, db in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                db = db.to(DEVICE)
                out = model(xb, grl_lambda=0.0)
                cls_loss = ce(out["logits"], yb)
                c_loss = center(out["embedding"], yb)
                d_loss = dom_ce(out["domain_logits"], db) if num_domains > 1 else torch.zeros((), device=DEVICE)
                loss = cls_loss + cfg.center_loss_weight * c_loss + cfg.domain_loss_weight * d_loss
                val_loss_sum += float(loss.detach().cpu()) * len(xb)
                y_true.extend(yb.detach().cpu().numpy().tolist())
                y_pred.extend(out["logits"].argmax(1).detach().cpu().numpy().tolist())

        val_metrics = compute_all_metrics(np.asarray(y_true), np.asarray(y_pred))
        train_acc = train_correct / max(1, train_total)
        train_loss = train_loss_sum / max(1, train_total)
        val_loss = val_loss_sum / max(1, len(y_true))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "grl_lambda": grl_lambda,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        # Validation score prioritizes balanced accuracy then macro-F1.
        score = val_metrics["balanced_accuracy"] + 0.20 * val_metrics["macro_f1"]
        improved = (score > best_val_score + 1e-5) or (val_loss < best_val_loss - 1e-4 and score >= best_val_score - 0.01)
        if improved:
            best_val_score = score
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_center = copy.deepcopy(center.state_dict())
            wait = 0
        else:
            wait += 1

        print(
            f"{fold_tag} | ep {epoch:02d}/{cfg.max_epochs} | "
            f"train {train_acc:.3f} | val {val_metrics['accuracy']:.3f} "
            f"bal {val_metrics['balanced_accuracy']:.3f} | "
            f"loss {val_loss:.4f} | GRL {grl_lambda:.3f}"
        )

        if wait >= cfg.patience:
            print(f"{fold_tag} | early stopping at epoch {epoch}")
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    center.load_state_dict(best_center)
    hist_df = pd.DataFrame(history)
    return model, center, hist_df

In [11]:
# ============================================================
# CELL 11 — INFERENCE + OPTIONAL CHECKPOINT HELPERS
# ============================================================
def predict_model(model, X, batch_size=256):
    model.eval()
    ds = TensorEEGOnlyDataset(X)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    preds = []
    probs = []
    embeddings = []
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(DEVICE)
            out = model(xb, grl_lambda=0.0)
            p = torch.softmax(out["logits"], dim=1)
            preds.append(p.argmax(1).cpu().numpy())
            probs.append(p.cpu().numpy())
            embeddings.append(out["embedding"].cpu().numpy())
    return np.concatenate(preds), np.concatenate(probs), np.concatenate(embeddings)


class TensorEEGOnlyDataset(Dataset):
    def __init__(self, X):
        self.X = np.asarray(X, dtype=np.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return torch.from_numpy(self.X[i])


def save_checkpoint(path, model, center, cfg, metadata):
    payload = {
        "model_state": model.state_dict(),
        "center_state": center.state_dict(),
        "config": asdict(cfg),
        "metadata": metadata,
    }
    torch.save(payload, path)


def load_checkpoint(path, num_domains):
    payload = torch.load(path, map_location=DEVICE)
    cfg_local = Config(**payload["config"])
    model = DGConvoReleNet(N_CLASSES, num_domains=num_domains, cfg=cfg_local).to(DEVICE)
    center = CenterLoss(N_CLASSES, cfg_local.embedding_dim, lr=cfg_local.center_loss_lr).to(DEVICE)
    model.load_state_dict(payload["model_state"])
    center.load_state_dict(payload["center_state"])
    return model, center, cfg_local, payload["metadata"]

## Strict protocol logic

For each within-dataset LOSO fold, the target subject is removed before **all** training/validation operations. Source subjects are split into grouped train/validation subjects; the target remains untouched until final test evaluation. For cross-dataset zero-calibration, the entire source dataset is used for training with an internal source-subject validation split, and every target subject is evaluated only at the end.

This follows the frozen Module 6 protocol: target epochs are never used for normalization or model selection.

In [12]:
# ============================================================
# CELL 12 — RUN ONE WITHIN-DATASET LOSO FOLD
# ============================================================
def run_within_fold(fold_row, fold_number):
    dataset = str(fold_row["dataset"])
    target_subject = str(fold_row["target_subject"])
    fold_id = int(fold_row["fold_id"])

    source_indices = safe_indices(fold_row["train_indices_json"])
    test_indices = safe_indices(fold_row["test_indices_json"])

    # Source-only grouped validation split.
    train_indices, val_indices, val_subjects = grouped_source_split(
        source_indices,
        cache_meta_df,
        target_subject=target_subject,
        seed=CFG.seed + CFG.validation_seed_offset + fold_id,
        val_fraction=CFG.validation_subject_fraction,
    )

    with HDF5Store(CACHE_PATH) as store:
        X_train_raw = store.get_X(train_indices)
        X_val_raw = store.get_X(val_indices)
        X_test_raw = store.get_X(test_indices)

    train_meta = cache_meta_df.loc[train_indices]
    val_meta = cache_meta_df.loc[val_indices]
    test_meta = cache_meta_df.loc[test_indices]

    # Strict source-only robust normalization.
    normalizer = SourceOnlyRobustNormalizer().fit(
        X_train_raw,
        train_meta["subject"].astype(str).tolist(),
    )
    normalizer.assert_target_excluded(target_subject)

    X_train = normalizer.transform(X_train_raw)
    X_val = normalizer.transform(X_val_raw)
    X_test = normalizer.transform(X_test_raw)

    y_train = compute_class_ids(train_meta)
    y_val = compute_class_ids(val_meta)
    y_test = compute_class_ids(test_meta)

    # Subject-adversarial domain labels are fit on training subjects only.
    train_domains, domain_map = make_domain_ids(train_meta)
    val_domains = np.asarray([domain_map.get(str(s), -1) for s in val_meta["subject"]], dtype=np.int64)
    # Validation subjects are intentionally not present in the domain head. For validation,
    # use a dummy domain ID that is ignored by the loss by setting domain loss weight to zero.
    # The classifier still sees validation data, but domain classification is not required.
    val_domains = np.zeros(len(val_domains), dtype=np.int64)

    num_domains = max(2, len(domain_map))

    # Target subject must have exactly one identity in the held-out test set.
    assert set(test_meta["subject"].astype(str)) == {target_subject}

    ckpt_path = CHECKPOINT_ROOT / f"within_{dataset}_{target_subject}.pt"
    history_path = HISTORY_ROOT / f"within_{dataset}_{target_subject}.csv"
    pred_path = PRED_ROOT / f"within_{dataset}_{target_subject}.csv"

    start = time.time()
    model, center, history = train_one_fold(
        X_train, y_train, train_domains,
        X_val, y_val, val_domains,
        num_domains=num_domains,
        cfg=CFG,
        seed=CFG.seed + fold_id,
        fold_tag=f"WITHIN {dataset}/{target_subject}",
    )

    preds, probs, embeddings = predict_model(model, X_test)
    metrics = compute_all_metrics(y_test, preds)

    row = {
        **metrics,
        "protocol": "within_dataset_loso",
        "dataset": dataset,
        "target_subject": target_subject,
        "fold_id": fold_id,
        "source_train_epochs": len(train_indices),
        "source_val_epochs": len(val_indices),
        "val_subjects": json.dumps(val_subjects),
        "runtime_sec": time.time() - start,
        "best_val_balanced_accuracy": float(history["val_balanced_accuracy"].max()),
        "best_val_accuracy": float(history["val_accuracy"].max()),
        "best_val_macro_f1": float(history["val_macro_f1"].max()),
        "epochs_run": len(history),
    }

    save_checkpoint(
        ckpt_path,
        model,
        center,
        CFG,
        {"protocol": row["protocol"], "dataset": dataset, "target_subject": target_subject,
         "fold_id": fold_id, "source_subject_count": len(domain_map)},
    )
    history.to_csv(history_path, index=False)

    if CFG.save_predictions:
        pred_df = test_meta[["cache_index", "dataset", "subject", "harmonized_class"]].copy()
        pred_df["true_id"] = y_test
        pred_df["pred_id"] = preds
        pred_df["pred_class"] = [ID_TO_CLASS[int(x)] for x in preds]
        for k in range(N_CLASSES):
            pred_df[f"prob_{ID_TO_CLASS[k]}"] = probs[:, k]
        pred_df.to_csv(pred_path, index=False)

    print("=" * 78)
    print(f"DONE | {dataset} | {target_subject} | accuracy={metrics['accuracy']:.4f} | balanced={metrics['balanced_accuracy']:.4f} | F1={metrics['macro_f1']:.4f}")
    print("Runtime sec:", round(time.time() - start, 2))
    return row

In [13]:
# ============================================================
# CELL 13 — FULL WITHIN-DATASET LOSO RUNNER (RESUMABLE)
# ============================================================
WITHIN_RESULT_PATH = MODULE8_ROOT / "dg_convorelenet_within_loso_results.csv"


def run_within_loso():
    folds = within_loso_df.copy()
    if CFG.smoke_test:
        folds = folds.groupby("dataset", group_keys=False).head(CFG.smoke_folds_per_protocol).copy()

    if WITHIN_RESULT_PATH.exists() and CFG.resume:
        existing = pd.read_csv(WITHIN_RESULT_PATH)
        done = set(existing["dataset"].astype(str) + "::" + existing["target_subject"].astype(str))
    else:
        existing = pd.DataFrame()
        done = set()

    rows = []
    for _, fold in folds.iterrows():
        key = str(fold["dataset"]) + "::" + str(fold["target_subject"])
        if key in done:
            print("SKIP existing:", key)
            continue
        row = run_within_fold(fold, int(fold["fold_id"]))
        rows.append(row)
        existing = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
        existing.to_csv(WITHIN_RESULT_PATH, index=False)

    return pd.read_csv(WITHIN_RESULT_PATH) if WITHIN_RESULT_PATH.exists() else pd.DataFrame(rows)

within_results = None
if CFG.run_within_loso:
    within_results = run_within_loso()
    print("Saved:", WITHIN_RESULT_PATH)
else:
    print("Within LOSO disabled.")

WITHIN BCI-IV-2a/S01 | ep 01/50 | train 0.387 | val 0.384 bal 0.384 | loss 1.7822 | GRL 0.000
WITHIN BCI-IV-2a/S01 | ep 02/50 | train 0.399 | val 0.491 bal 0.491 | loss 1.6954 | GRL 0.000
WITHIN BCI-IV-2a/S01 | ep 03/50 | train 0.454 | val 0.468 bal 0.468 | loss 1.6412 | GRL 0.000
WITHIN BCI-IV-2a/S01 | ep 04/50 | train 0.530 | val 0.671 bal 0.671 | loss 1.4427 | GRL 0.000
WITHIN BCI-IV-2a/S01 | ep 05/50 | train 0.576 | val 0.593 bal 0.593 | loss 1.4888 | GRL 0.000
WITHIN BCI-IV-2a/S01 | ep 06/50 | train 0.652 | val 0.662 bal 0.662 | loss 1.4277 | GRL -0.197
WITHIN BCI-IV-2a/S01 | ep 07/50 | train 0.729 | val 0.606 bal 0.606 | loss 1.5241 | GRL -0.196
WITHIN BCI-IV-2a/S01 | ep 08/50 | train 0.741 | val 0.611 bal 0.611 | loss 1.5595 | GRL -0.195
WITHIN BCI-IV-2a/S01 | ep 09/50 | train 0.789 | val 0.648 bal 0.648 | loss 1.5362 | GRL -0.193
WITHIN BCI-IV-2a/S01 | ep 10/50 | train 0.820 | val 0.560 bal 0.560 | loss 1.6368 | GRL -0.192
WITHIN BCI-IV-2a/S01 | ep 11/50 | train 0.871 | val 0.6

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 14 — CROSS-DATASET ZERO-CALIBRATION RUNNER
# ============================================================
TRANSFER_RESULT_PATH = MODULE8_ROOT / "dg_convorelenet_cross_dataset_zero_calibration_results.csv"


def train_source_direction(source_dataset, direction_folds):
    first_fold = direction_folds.iloc[0]
    source_indices = safe_indices(first_fold["train_indices_json"])

    train_indices, val_indices, val_subjects = grouped_source_split(
        source_indices,
        cache_meta_df,
        seed=CFG.seed + hash(source_dataset) % 10000,
        val_fraction=CFG.validation_subject_fraction,
    )

    with HDF5Store(CACHE_PATH) as store:
        X_train_raw = store.get_X(train_indices)
        X_val_raw = store.get_X(val_indices)

    train_meta = cache_meta_df.loc[train_indices]
    val_meta = cache_meta_df.loc[val_indices]

    normalizer = SourceOnlyRobustNormalizer().fit(
        X_train_raw,
        train_meta["subject"].astype(str).tolist(),
    )
    X_train = normalizer.transform(X_train_raw)
    X_val = normalizer.transform(X_val_raw)

    y_train = compute_class_ids(train_meta)
    y_val = compute_class_ids(val_meta)

    train_domains, domain_map = make_domain_ids(train_meta)
    val_domains = np.zeros(len(val_meta), dtype=np.int64)

    ckpt_path = CHECKPOINT_ROOT / f"transfer_source_{source_dataset}.pt"
    history_path = HISTORY_ROOT / f"transfer_source_{source_dataset}.csv"

    print("=" * 78)
    print("TRAIN SOURCE MODEL:", source_dataset)
    print("Train epochs:", len(train_indices), "Val epochs:", len(val_indices))
    print("Validation subjects:", val_subjects)

    model, center, history = train_one_fold(
        X_train, y_train, train_domains,
        X_val, y_val, val_domains,
        num_domains=max(2, len(domain_map)),
        cfg=CFG,
        seed=CFG.seed + 5000,
        fold_tag=f"TRANSFER {source_dataset}",
    )

    save_checkpoint(
        ckpt_path,
        model,
        center,
        CFG,
        {"protocol": "cross_dataset_zero_calibration", "source_dataset": source_dataset,
         "source_subject_count": len(domain_map)},
    )
    history.to_csv(history_path, index=False)

    return model, center, normalizer


def run_cross_dataset():
    folds = transfer_df.copy()
    if CFG.smoke_test:
        folds = folds.groupby(["source_dataset", "target_dataset"], group_keys=False).head(CFG.smoke_folds_per_protocol).copy()

    if TRANSFER_RESULT_PATH.exists() and CFG.resume:
        existing = pd.read_csv(TRANSFER_RESULT_PATH)
        done = set(existing["source_dataset"].astype(str) + "->" + existing["target_dataset"].astype(str) + "::" + existing["target_subject"].astype(str))
    else:
        existing = pd.DataFrame()
        done = set()

    source_models = {}
    for source_dataset in sorted(folds["source_dataset"].unique()):
        direction_folds = folds[folds["source_dataset"] == source_dataset].copy()
        target_dataset = str(direction_folds.iloc[0]["target_dataset"])
        if (source_dataset, target_dataset) in source_models:
            continue
        source_models[(source_dataset, target_dataset)] = train_source_direction(source_dataset, direction_folds)

    with HDF5Store(CACHE_PATH) as store:
        for _, fold in folds.iterrows():
            source_dataset = str(fold["source_dataset"])
            target_dataset = str(fold["target_dataset"])
            target_subject = str(fold["target_subject"])
            key = source_dataset + "->" + target_dataset + "::" + target_subject
            if key in done:
                print("SKIP existing:", key)
                continue

            model, center, normalizer = source_models[(source_dataset, target_dataset)]
            test_indices = safe_indices(fold["test_indices_json"])
            X_test_raw = store.get_X(test_indices)
            test_meta = cache_meta_df.loc[test_indices]
            y_test = compute_class_ids(test_meta)

            normalizer.assert_target_excluded(target_subject)
            X_test = normalizer.transform(X_test_raw)
            preds, probs, embeddings = predict_model(model, X_test)
            metrics = compute_all_metrics(y_test, preds)

            row = {
                **metrics,
                "protocol": "cross_dataset_zero_calibration",
                "source_dataset": source_dataset,
                "target_dataset": target_dataset,
                "target_subject": target_subject,
                "fold_id": int(fold["fold_id"]),
                "target_epochs": len(test_indices),
            }
            existing = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
            existing.to_csv(TRANSFER_RESULT_PATH, index=False)

            if CFG.save_predictions:
                safe_name = f"transfer_{source_dataset}_to_{target_dataset}_{target_subject}.csv"
                pred_df = test_meta[["cache_index", "dataset", "subject", "harmonized_class"]].copy()
                pred_df["true_id"] = y_test
                pred_df["pred_id"] = preds
                pred_df["pred_class"] = [ID_TO_CLASS[int(x)] for x in preds]
                for k in range(N_CLASSES):
                    pred_df[f"prob_{ID_TO_CLASS[k]}"] = probs[:, k]
                pred_df.to_csv(PRED_ROOT / safe_name, index=False)

            print(f"TRANSFER | {source_dataset} -> {target_dataset} | {target_subject} | Acc={metrics['accuracy']:.4f} | Bal={metrics['balanced_accuracy']:.4f} | F1={metrics['macro_f1']:.4f}")

    return pd.read_csv(TRANSFER_RESULT_PATH) if TRANSFER_RESULT_PATH.exists() else pd.DataFrame()

transfer_results = None
if CFG.run_cross_dataset:
    transfer_results = run_cross_dataset()
    print("Saved:", TRANSFER_RESULT_PATH)
else:
    print("Cross-dataset disabled.")

In [ ]:
# ============================================================
# CELL 15 — RESULTS SUMMARY + PER-SUBJECT DISTRIBUTION
# ============================================================
def summarize_results(df, title):
    if df is None or len(df) == 0:
        print(title, ": no results")
        return None
    metrics = ["accuracy", "balanced_accuracy", "macro_f1"]
    summary = pd.DataFrame({
        "metric": metrics,
        "mean": [df[m].mean() for m in metrics],
        "std": [df[m].std(ddof=1) for m in metrics],
        "median": [df[m].median() for m in metrics],
        "min": [df[m].min() for m in metrics],
        "max": [df[m].max() for m in metrics],
    })
    print("\n" + "="*78)
    print(title)
    print("="*78)
    display(summary)
    print("Accuracy mean ± std: %.2f ± %.2f %%" % (100*df["accuracy"].mean(), 100*df["accuracy"].std(ddof=1)))
    print("Balanced accuracy : %.2f %%" % (100*df["balanced_accuracy"].mean()))
    print("Macro F1           : %.2f %%" % (100*df["macro_f1"].mean()))
    return summary

within_summary = summarize_results(within_results, "DG-ConvoReleNet — WITHIN-DATASET LOSO")
transfer_summary = summarize_results(transfer_results, "DG-ConvoReleNet — CROSS-DATASET ZERO-CALIBRATION")

if within_results is not None and len(within_results):
    by_dataset = within_results.groupby("dataset")["accuracy"].agg(["mean", "std", "min", "max", "count"])
    print("\nWithin-dataset by dataset:")
    display(100 * by_dataset)

if transfer_results is not None and len(transfer_results):
    by_direction = transfer_results.groupby(["source_dataset", "target_dataset"])["accuracy"].agg(["mean", "std", "min", "max", "count"])
    print("\nCross-dataset by direction:")
    display(100 * by_direction)

In [ ]:
# ============================================================
# CELL 16 — PLOT TRAINING HISTORY EXAMPLES
# ============================================================
if CFG.plot_histories:
    history_files = sorted(HISTORY_ROOT.glob("within_*.csv"))
    if history_files:
        # Plot the first completed fold and one more representative fold if available.
        chosen = history_files[:min(2, len(history_files))]
        for hp in chosen:
            h = pd.read_csv(hp)
            fig = plt.figure(figsize=(8, 4))
            plt.plot(h["epoch"], h["train_accuracy"], label="Train")
            plt.plot(h["epoch"], h["val_accuracy"], label="Validation")
            plt.xlabel("Epoch")
            plt.ylabel("Accuracy")
            plt.title(hp.stem)
            plt.legend()
            plt.tight_layout()
            plt.show()

In [ ]:
# ============================================================
# CELL 17A — SAVE REPRODUCIBILITY / MODEL SPECIFICATION
# ============================================================
SPEC_OUTPUT = MODULE8_ROOT / "module_8_model_and_protocol_specification.json"

spec = {
    "module": 8,
    "model_name": "DG-ConvoReleNet",
    "input_shape": [22, 640],
    "sampling_rate_hz": 160.0,
    "primary_band_hz": [8.0, 30.0],
    "classes": PRIMARY_CLASSES,
    "epochs_total": int(len(cache_meta_df)),
    "subjects_total": int(cache_meta_df["subject"].nunique()),
    "strict_protocols": ["within_dataset_loso", "cross_dataset_zero_calibration"],
    "normalization": "source-only channel-wise median/IQR",
    "architecture": {
        "temporal_kernels": [15, 31, 63],
        "temporal_channels_each": CFG.temporal_channels_each,
        "spatial_channels": CFG.spatial_channels,
        "token_dim": CFG.token_dim,
        "transformer_layers": CFG.transformer_layers,
        "transformer_heads": CFG.transformer_heads,
        "transformer_ffn": CFG.transformer_ffn,
        "attention_pooling": True,
        "embedding_dim": CFG.embedding_dim,
        "activation": "Tanh",
        "lstm": CFG.use_lstm,
    },
    "loss": {
        "classification": "CrossEntropy(label_smoothing)",
        "center_loss_weight": CFG.center_loss_weight,
        "subject_adversarial_domain_loss_weight": CFG.domain_loss_weight,
        "grl_max_lambda": CFG.grl_max_lambda,
    },
    "optimization": {
        "optimizer": "AdamW",
        "learning_rate": CFG.lr,
        "weight_decay": CFG.weight_decay,
        "scheduler": "CosineAnnealingLR",
        "gradient_clip": CFG.grad_clip,
        "early_stopping_patience": CFG.patience,
    },
    "training_only_augmentation": True,
    "target_data_used_for_model_selection": False,
    "target_statistics_used": False,
    "seed": CFG.seed,
}
with open(SPEC_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(spec, f, indent=2)
print("Saved specification:", SPEC_OUTPUT)


In [ ]:
# ============================================================
# CELL 17 — FINAL QA / LEAKAGE AUDIT / ARTIFACTS
# ============================================================
print("=" * 78)
print("MODULE 8 FINAL QA")
print("=" * 78)

assert CACHE_PATH.exists()
assert set(PRIMARY_CLASSES) == set(cache_meta_df["harmonized_class"].unique())
assert cache_meta_df["subject"].nunique() == 118

if within_results is not None and len(within_results):
    assert within_results["target_subject"].notna().all()
    assert within_results["accuracy"].between(0, 1).all()

if transfer_results is not None and len(transfer_results):
    assert transfer_results["target_subject"].notna().all()
    assert transfer_results["accuracy"].between(0, 1).all()

print("✓ Input cache: frozen Module 5/6 interface")
print("✓ Source-only robust normalization")
print("✓ Grouped source validation")
print("✓ Target excluded from strict LOSO training and normalization")
print("✓ Center loss enabled")
print("✓ Subject-adversarial domain loss enabled")
print("✓ Multi-scale CNN + spatial CNN + Transformer + attention pooling")
print("✓ 128-D embedding")
print("✓ Tanh activation")
print("✓ Optional LSTM is OFF by default")
print("\nAll configured checks passed.")

## Optional next experiment — few-shot adaptation

Do **not** mix target calibration into the strict results. The uploaded 2026 transfer-learning study supports conservative fine-tuning, with lower learning rates and fewer epochs used to reduce catastrophic forgetting. A separate notebook mode can later fine-tune only on an explicitly declared calibration subset (for example 5% or 10% of target trials) and report it as a separate *few-shot adaptation* experiment.